In [2]:
import pandas as pd
import numpy as np
!pip install openpyxl

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\adhamz\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [3]:
df_400m = pd.read_csv('400m.csv')
print(df_400m.shape)
df_400m.head()

(309, 24)


,Date,About,by,Session,Pool Length,Stroke,Target Time,Test Time,Heart Rate,Stroke Count Check 1,Stroke Count Check 2,RPE,Lactate,Test Rating,Comments,Athlete_Date_Key,Target Time Seconds,Test Time Seconds,RPE Value,Test Rating Value,Test to Target Time Difference,Stroke Count Check 2 to Stroke Count Check 1 Difference,event-uuid,group-uuid
0,06-07-2026,George Smith,Adrian Campbell,AM,LC,Freestyle,4:31.2,4:34.0,176.0,23.0,25.0,6.0,NaN,NaN,No test for 2 weeks. Metrics pretty stable,George Smith-06/07/2026,271.2,274.0,6.0,NaN,2.8,2.0,ab8490c9-3397-4c56-b473-47e919420dba,3d2938ad-1d28-48dc-8841-d8e36c01e4bc
1,06-07-2026,Holly McGill,Adrian Campbell,AM,LC,Freestyle,4:58.7,4:54.7,193.0,33.0,34.0,6.0,NaN,NaN,"No test for 2 weeks. All fairly stable, albei...",Holly McGill-06/07/2026,298.7,294.7,6.0,NaN,-4.0,1.0,44173e0f-a8a0-48af-86a0-e508306c3516,cc1bc5bd-5d83-4b12-9a42-8d64174f8bc9
2,06-07-2026,Duncan Scott,Adrian Campbell,AM,LC,Freestyle,4:38.9,4:39.2,152.0,22.0,24.0,4.5,NaN,NaN,"New pacing (+6sec). Manual HR. SC -7, RPE -1.5",Duncan Scott-06/07/2026,278.9,279.2,5.0,NaN,0.3,2.0,8559ea9a-8a01-479d-993c-36edd08e5d53,244f62d0-d02a-40f7-9412-c78c991d040f
3,06-07-2026,Evelyn Davis,Adrian Campbell,AM,LC,Freestyle,4:58.2,5:03.8,163.0,30.0,32.0,4.5,NaN,NaN,"No test for 2 weeks. Swim Ttime +5sec, but HR...",Evelyn Davis-06/07/2026,298.2,303.8,5.0,NaN,5.6,2.0,26a5dbb2-0019-4c9e-9721-cb977b6e1951,599d108c-3ee0-4788-86fb-a1071e16bd96
4,06-07-2026,Katie Shanahan,Adrian Campbell,AM,LC,Freestyle,4:52.7,4:49.0,178.0,27.0,29.0,4.5,NaN,NaN,No test for 2 weeks. Numbers all good and loo...,Katie Shanahan-06/07/2026,292.7,289.0,5.0,NaN,-3.7,2.0,bee4fc7f-80d4-453f-bcc5-6971c44e141b,d578072c-23a9-4010-915d-23b2e519d3b5


In [5]:
def run_audit(df, dataset_name, athlete_col='About', date_col='Date',
              missing_threshold=70, constant_threshold=1, id_threshold=95):
    
    """
    Runs a standardised audit on a dataset and saves results to Excel.
    
    Parameters:
    
    df
        The dataset to audit
    dataset_name
        Used for the output filename 
    athlete_col
        Column containing athlete identifier (default: 'About')
    date_col
        Column containing date (default: 'Date')
    missing_threshold
        % missing, flags it as high missing (flag_high_missing = TRUE) (default: over 70)
    constant_threshold
        % unique, flags it as a likely constant (flag_near_constant = TRUE) (default: under 1)
    id_threshold : 
        % unique, flags it as a likely identifier (flag_identifier = TRUE) (default: over 95)
    """

    
    ##Variable Summary (This produces sheet 1):
    
    n = len(df) #total number of rows, for calculations later
    rows = [] #Creates an empty list to loop through every column 

    for col in df.columns:
        n_missing = df[col].isnull().sum()
        pct_missing = round(n_missing / n * 100, 2)
        pct_unique = round(df[col].nunique() / n * 100, 2)

        flag_high_missing = pct_missing >= missing_threshold
        flag_near_constant = pct_unique < constant_threshold
        flag_identifier = pct_unique >= id_threshold
        flag_structurally_empty = pct_missing == 100

        #Identifier detection
        is_object_dtype = df[col].dtype == 'object' or pd.api.types.is_string_dtype(df[col])
        is_low_missing = pct_missing < 5
        is_low_unique = pct_unique <10

        is_likely_categorical_identifier = (    #categorical, almost always present, low uniqueness (e.g. names)
            is_object_dtype and
            is_low_missing and
            is_low_unique and
            df[col].nunique() > 5 ) #fewer than 5 unique values is likely a feature

        is_high_cardinality_identifier = pct_unique >= id_threshold #nearly every value unique (IDs)

        is_keyword_identifier = (
            'uuid' in col.lower() or
            col.lower().endswith('_id') or
            col.lower().endswith('_datetime') or
            col.lower().endswith('_date_time'))

        #Auto-assign variable role
        if col == date_col or col == athlete_col:
            role = 'Identifier'
        elif flag_structurally_empty:
            role = 'Exclude (structurally empty)'
        elif is_likely_categorical_identifier or is_high_cardinality_identifier or is_keyword_identifier:
            role = 'Identifier'
        elif is_object_dtype and df[col].nunique()<=5 and is_low_missing:
            role = 'Sparse categorical - review'
        elif flag_high_missing:
            role = 'Review (high missingness)'
        elif flag_near_constant:
            #Check if variable is still informative vs truly constant
            non_missing = df[col].dropna()
            non_zero = non_missing[non_missing != 0]
            if len(non_zero) > 0 and non_zero.nunique() > 3:
                role = 'Sparse - Review'
            else:
                role = 'Exclude (near constant)'
        else:
            role = 'Predictor candidate'

        rows.append({
            'variable': col,
            'dtype': str(df[col].dtype),
            'n_missing': n_missing,
            'pct_missing': pct_missing,
            'pct_unique': pct_unique,
            'flag_high_missing': flag_high_missing,
            'flag_near_constant': flag_near_constant,
            'flag_identifier': flag_identifier,
            'flag_structurally_empty': flag_structurally_empty,
            'variable_role': role,})

    var_summary = pd.DataFrame(rows)

    ##Temporal Evaluation (this produces Sheet 2):
    
    temporal_row = {} #creates empty dictionary to fill with temporal measures

    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce', dayfirst=True)
        temporal_row['min_date'] = df[date_col].min()
        temporal_row['max_date'] = df[date_col].max()
    else:
        temporal_row['min_date'] = 'Date column not found'
        temporal_row['max_date'] = 'Date column not found'

    if athlete_col in df.columns:
        temporal_row['n_athletes'] = df[athlete_col].nunique() #number of athletes

        #athlete observation counts
        obs_per_athlete = df.groupby(athlete_col).size()
        temporal_row['mean_obs_per_athlete']   = round(obs_per_athlete.mean(), 2) #round to two decimal places
        temporal_row['median_obs_per_athlete'] = obs_per_athlete.median()
        temporal_row['min_obs_per_athlete']    = obs_per_athlete.min()
        temporal_row['max_obs_per_athlete']    = obs_per_athlete.max()

        #athlete gap between consecutive observations
        if date_col in df.columns:
            gaps = (
                df.sort_values([athlete_col, date_col])
                .groupby(athlete_col)[date_col]
                .diff()
                .dt.days
                .dropna() )
            temporal_row['mean_days_between_obs']   = round(gaps.mean(), 2)
            temporal_row['min_days_between_obs']    = gaps.min()
            temporal_row['max_days_between_obs']    = gaps.max()
        else:
            temporal_row['mean_days_between_obs'] = 'N/A'
            temporal_row['min_days_between_obs']  = 'N/A'
            temporal_row['max_days_between_obs']  = 'N/A'
    else: #if wrong column name is enetered
        temporal_row['n_athletes'] = 'Athlete column not found'
        for k in ['mean_obs_per_athlete','median_obs_per_athlete',
                  'min_obs_per_athlete','max_obs_per_athlete',
                  'mean_days_between_obs','min_days_between_obs',
                  'max_days_between_obs']:
            temporal_row[k] = 'N/A'

    ##auto-generate interpretation
    
    try:
        interp = (
            f"{dataset_name} audit covers {temporal_row['n_athletes']} athletes "
            f"from {str(temporal_row['min_date'])[:10]} to "
            f"{str(temporal_row['max_date'])[:10]}. "
            f"Mean observations per athlete: {temporal_row['mean_obs_per_athlete']} "
            f"(range {temporal_row['min_obs_per_athlete']}–"
            f"{temporal_row['max_obs_per_athlete']}). "
            f"Mean gap between observations: {temporal_row['mean_days_between_obs']} days "
            f"(range {temporal_row['min_days_between_obs']}–"
            f"{temporal_row['max_days_between_obs']} days).")
    except Exception:
        interp = 'Could not auto-generate interpretation — check column names.'

    temporal_row['interpretation'] = interp
    temporal_df = pd.DataFrame([temporal_row])

    ##summary counts for quick reference
    
    role_counts = var_summary['variable_role'].value_counts()
    print(f"\n{'='*50}") #creates a visual divider line
    print(f"AUDIT COMPLETE: {dataset_name}")
    print(f"{'='*50}")
    print(f"Total variables: {len(var_summary)}")
    print(f"Total rows: {n}")
    print(f"\nVariable role breakdown:")
    for role, count in role_counts.items(): #loops through each category and it's count
        print(f"  {role:<40} {count}") #"40" makes it 40 characters wide to create neat column
    print(f"\nTemporal coverage:")
    print(f"Athletes: {temporal_row['n_athletes']}")
    print(f"Date range: {str(temporal_row['min_date'])[:10]} to {str(temporal_row['max_date'])[:10]}") #"10" takes the first 10 characters (just yyyy-mm-dd)
    print(f"Mean obs/athlete: {temporal_row['mean_obs_per_athlete']}")
    print(f"Mean gap (days): {temporal_row['mean_days_between_obs']}")

    ##save to Excel 
    
    filename = f"{dataset_name.replace(' ', '_')}_audit.xlsx"
    with pd.ExcelWriter(filename, engine='openpyxl') as writer: #"openxyl" enables reading and writing excel files
        var_summary.to_excel(writer, sheet_name='Variable Summary', index=False)
        temporal_df.to_excel(writer, sheet_name='Temporal Evaluation', index=False)

    print(f"\nSaved: {filename}")
    return var_summary, temporal_df

In [6]:
var_summary, temporal = run_audit(
    df=df_400m,
    dataset_name='400m Weekly Test',
    athlete_col='About',
    date_col='Date'
)


AUDIT COMPLETE: 400m Weekly Test
Total variables: 24
Total rows: 309

Variable role breakdown:
  Predictor candidate                      11
  Identifier                               6
  Sparse categorical - review              4
  Review (high missingness)                3

Temporal coverage:
Athletes: 16
Date range: 2025-11-10 to 2026-07-06
Mean obs/athlete: 19.31
Mean gap (days): 12.18

Saved: 400m_Weekly_Test_audit.xlsx


In [7]:
#check what the 15 predictor candidates actually are
print(f"the predictor candidates are", var_summary[var_summary['variable_role'] == 'Predictor candidate']['variable'].tolist())

#check what the 13 identifiers are
print(f"the identifiers are", var_summary[var_summary['variable_role'] == 'Identifier']['variable'].tolist())

the predictor candidates are ['Test Time', 'Heart Rate', 'Stroke Count Check 1', 'Stroke Count Check 2', 'RPE', 'Comments', 'Target Time Seconds', 'Test Time Seconds', 'RPE Value', 'Test to Target Time Difference', 'Stroke Count Check 2 to Stroke Count Check 1 Difference']
the identifiers are ['Date', 'About', 'Target Time', 'Athlete_Date_Key', 'event-uuid', 'group-uuid']
